In [5]:
from selenium import webdriver
from selenium . webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager

from selenium.webdriver.chrome.options import Options


opts = Options()
opts.add_argument("--headless")           # use the modern headless mode
opts.add_argument("--no-sandbox")         # handy for CI or Docker
opts.add_argument("--disable-dev-shm-usage")


In [6]:
driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()))
driver.get("https://fbref.com/en/comps/9/Premier-League-Stats")

In [7]:
result = []
final_result = []

content = driver.find_element(By.ID, "div_results2025-202691_overall")

In [8]:
buttons = driver.find_elements(By.XPATH, ".//td[@data-stat='team']/a")

In [18]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)
league_url = driver.current_url

buttons = driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")
hrefs = [a.get_attribute('href') for a in buttons]

for href in hrefs:
    driver.get(href)
    table = wait.until(EC.presence_of_element_located((By.ID, "all_stats_standard")))
    result.append(table.get_attribute('outerHTML'))

    driver.get(league_url)
    wait.until(EC.presence_of_element_located((By.ID, 'div_results2025-202691_overall')))


matchlogs = []
matchlogs_teams = []
buttons = driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")
hrefs = [a.get_attribute('href') for a in buttons]
for href in hrefs:
    driver.get(href)
    tbl = wait.until(EC.presence_of_element_located((By.ID, "matchlogs_for")))
    matchlogs.append(tbl.get_attribute('outerHTML'))

    team_name = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1'))).text
    matchlogs_teams.append(team_name)

    driver.get(league_url)
    wait.until(EC.presence_of_element_located((By.ID, 'div_results2025-202691_overall')))

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=50975): Read timed out. (read timeout=120)

In [ ]:
import pandas as pd
from io import StringIO



dfs = []
for html in result:
    if not html:
        continue
    try:

        df = pd.read_html(StringIO(html))[0]
        dfs.append(df)
    except Exception:
      
        continue

combined = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

combined


dfs_match = []
for i, html in enumerate(matchlogs):
    if not html:
        continue
    df = pd.read_html(StringIO(html))[0]

    if df.shape[1] > 2:
        df = df.iloc[:, :-2]

    keep = [c for c in df.columns if df[c].notna().sum() >= 3]
    df = df[keep]

    team = matchlogs_teams[i] if i < len(matchlogs_teams) else None
    df['team'] = team
    dfs_match.append(df)
combined_matchlogs = pd.concat(dfs_match, ignore_index=True) if dfs_match else pd.DataFrame()

combined_matchlogs

""


In [15]:
print(combined)

Empty DataFrame
Columns: []
Index: []
